# zero_ref_check — contribution of the complete reference-conditioned package (reference images plus their pointer sentences) + reference-subject-count axis
0-ref (text-to-image) vs 488 baseline, joined on uid (baseline row order differs from test.jsonl, so pairing is by uid not position).
SC drop = contribution of the reference package (images plus pointer text, both removed together); PF at 0-ref = base text-to-image floor, stored for the adapter-vs-base drift check post-LoRA.
Reference-subject-count axis (1/2/3 by len(input_images)) and person/object provenance from the baseline; no new generation.

In [ ]:
from google.colab import drive
import os, json, math, statistics as st, re
from collections import defaultdict
from scipy.stats import ttest_rel, t as tdist
drive.mount("/content/drive")

DRIVE  = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
SCORES = f"{DRIVE}/scores"

def load_by_uid(path):
    d = {}
    for l in open(path, encoding="utf-8"):
        r = json.loads(l)
        if "error" in r: continue
        d[r["uid"]] = r
    return d

zr = load_by_uid(f"{SCORES}/test__zero_ref.jsonl")   # 0-ref (text-to-image)
bl = load_by_uid(f"{SCORES}/test__baseline.jsonl")   # 488 baseline (with references)
meta = {r["output_image"] + "|" + "|".join(sorted(r["input_images"])): r
        for r in [json.loads(l) for l in open(f"{DRIVE}/test.jsonl", encoding="utf-8")]}
I = sorted(set(zr) & set(bl))
assert I, "no shared uids between 0-ref and baseline"
print(f"0-ref {len(zr)} | baseline {len(bl)} | test.jsonl {len(meta)} | uid-paired {len(I)}")

Mounted at /content/drive
0-ref 488 | baseline 488 | test.jsonl 488 | uid-paired 488


## Reference-package contribution (images plus pointer text) — 0-ref vs baseline (uid-paired)

In [ ]:
ov = lambda d: math.sqrt(max(d["PF"], 0) * max(d["SC"], 0))

def paired(metric):
    d = [metric(zr[u]) - metric(bl[u]) for u in I]
    n = len(d); mdv = st.mean(d); se = st.stdev(d) / math.sqrt(n)
    t, p = ttest_rel([metric(zr[u]) for u in I], [metric(bl[u]) for u in I])
    tc = tdist.ppf(0.95, n - 1)
    return mdv, p, mdv - tc*se, mdv + tc*se

print(f"0-ref minus baseline (uid-paired, n={len(I)}):")
for name, m in [("Overall", ov), ("SC", lambda d: d["SC"]), ("PF", lambda d: d["PF"])]:
    mdv, p, lo, hi = paired(m)
    print(f"  {name:8s}  delta {mdv:+.3f}  p {p:.3g}  90%CI [{lo:+.3f}, {hi:+.3f}]")

blSC = st.mean([bl[u]["SC"] for u in I]); zrSC = st.mean([zr[u]["SC"] for u in I])
blPF = st.mean([bl[u]["PF"] for u in I]); zrPF = st.mean([zr[u]["PF"] for u in I])
print(f"\nreference-package contribution to SC (images + pointer text) = baseline {blSC:.3f} - 0-ref {zrSC:.3f} = {blSC - zrSC:+.3f}")
print(f"base T2I floor (PF at 0-ref) = {zrPF:.3f}   (baseline PF {blPF:.3f})   [store for adapter-vs-base drift check]")

0-ref minus baseline (uid-paired, n=488):
  Overall   delta -2.236  p 3.69e-63  90%CI [-2.424, -2.047]
  SC        delta -3.053  p 5.92e-79  90%CI [-3.274, -2.833]
  PF        delta -0.766  p 1.1e-16  90%CI [-0.913, -0.620]

reference-package contribution to SC (images + pointer text) = baseline 8.088 - 0-ref 5.035 = +3.053
base T2I floor (PF at 0-ref) = 7.238   (baseline PF 8.004)   [store for adapter-vs-base drift check]


## Reference-subject-count axis and provenance (from baseline; no new generation)
Count = len(input_images) per row (single/double/triple are packaging directories, not counts).

In [ ]:
# person vs object per reference: head noun of each pointing-clause subject, taken before any preposition.
# heuristic — free-text subjects are inherently fuzzy; the person>object SC gap holds across every classifier variant tried,
# so the finding does not hinge on this. (An earlier substring version mis-hit "person" in "personal", "hair" in "chair".)
PERSON = {"man","woman","men","women","person","people","boy","girl","boys","girls","guy","lady","businessman",
          "businesswoman","musician","musicians","child","children","kid","kids","baby","model","player","athlete",
          "worker","couple","gentleman","barber","tourist","tourists","doctor","doctors","assistant","assistants"}
_PREP = (r"\b(in|with|on|at|by|of|from|and|to|dressed|wearing|holding|using|reviewing|sitting|standing|"
         r"typing|sketching|working|reading|writing|playing|looking|smiling|drinking|reaching|leaning|posing|walking|browsing)\b")
def person_obj(instr):
    kinds = set()
    for p in re.split(r"(?<=\.)\s+", instr):
        if "<img>" not in p: continue
        m = re.search(r"(?:the\s+)?(.+?)\s+(?:is|are)\s+", p.lower())
        head = re.split(_PREP, m.group(1) if m else p.lower())[0].strip().split()
        kinds.add("person" if (head and head[-1] in PERSON) else "object")
    return "person" if kinds == {"person"} else ("object" if kinds == {"object"} else "mixed")

def report(by_uid, tag):
    print(f"\n{tag} - reference-subject-count axis (by len(input_images)):")
    byc = defaultdict(list)
    for u in I: byc[len(meta[u]["input_images"])].append(by_uid[u])
    for n in (1, 2, 3):
        g = byc[n]
        print(f"  {n}-ref  PF {st.mean([x['PF'] for x in g]):.3f}  SC {st.mean([x['SC'] for x in g]):.3f}  Overall {st.mean([ov(x) for x in g]):.3f}  (n={len(g)})")
    print(f"{tag} - provenance (person vs object, heuristic from instruction):")
    byp = defaultdict(list)
    for u in I: byp[person_obj(meta[u]["instruction"])].append(by_uid[u])
    for k in ["person", "object", "mixed"]:
        g = byp.get(k, [])
        if g: print(f"  {k:12s} PF {st.mean([x['PF'] for x in g]):.3f}  SC {st.mean([x['SC'] for x in g]):.3f}  Overall {st.mean([ov(x) for x in g]):.3f}  (n={len(g)})")

report(bl, "baseline (with references)")
report(zr, "0-ref (text-to-image)")


baseline (with references) - reference-subject-count axis (by len(input_images)):
  1-ref  PF 8.181  SC 8.058  Overall 7.994  (n=243)
  2-ref  PF 7.851  SC 8.155  Overall 7.958  (n=194)
  3-ref  PF 7.745  SC 7.980  Overall 7.832  (n=51)
baseline (with references) - provenance (person vs object, heuristic from instruction):
  person       PF 8.254  SC 8.931  Overall 8.557  (n=130)
  object       PF 7.853  SC 7.415  Overall 7.493  (n=224)
  mixed        PF 8.015  SC 8.396  Overall 8.173  (n=134)

0-ref (text-to-image) - reference-subject-count axis (by len(input_images)):
  1-ref  PF 7.782  SC 4.757  Overall 5.644  (n=243)
  2-ref  PF 6.773  SC 5.284  Overall 5.810  (n=194)
  3-ref  PF 6.412  SC 5.412  Overall 5.810  (n=51)
0-ref (text-to-image) - provenance (person vs object, heuristic from instruction):
  person       PF 8.023  SC 5.669  Overall 6.429  (n=130)
  object       PF 7.201  SC 4.442  Overall 5.292  (n=224)
  mixed        PF 6.537  SC 5.410  Overall 5.775  (n=134)
